# 02 - Supervised & Deep Learning Model Benchmarking
## AI-Based Cyber Risk Detection System

This notebook benchmarks and compares supervised algorithms:
- **Logistic Regression**
- **Gaussian Naive Bayes**
- **K-Nearest Neighbours (KNN)**
- **Support Vector Machine (SVM)**
- **Decision Tree & Random Forest**
- **Multilayer Perceptron (Scikit-Learn MLP)**
- **Native PyTorch Deep Neural Network (DNN)**
- **Ensemble Architectures (Voting, Stacking, Bagging)**
- **Optuna Bayesian Hyperparameter Optimization**

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.pipeline import CyberRiskPipeline
from src.models.supervised import SupervisedModelFactory, ModelTrainer
from src.models.pytorch_net import PyTorchRiskClassifier
from src.models.ensemble import EnsembleBuilder
from src.models.evaluation import ModelEvaluator
from src.utils.visualization import plot_confusion_matrix, plot_model_comparison

print("Model benchmarking modules ready!")

### 1. Preparing Training & Validation Splits

In [ ]:
pipeline = CyberRiskPipeline()
pipeline.run_data_pipeline(use_synthetic=True)
print(f"X_train shape: {pipeline.X_train.shape}, X_test shape: {pipeline.X_test.shape}")

### 2. Training All Supervised Classifiers (Including PyTorch DNN)

In [ ]:
factory = SupervisedModelFactory()
models = factory.create_all_enabled()
trainer = ModelTrainer()

trained_models = {}
for name, model in models.items():
    print(f"Training {name}...")
    trained_models[name] = trainer.train(model, pipeline.X_train, pipeline.y_train)

# Add Bagging Ensemble
ensemble_builder = EnsembleBuilder()
bagging = ensemble_builder.build_bagging()
if bagging is not None:
    bagging.fit(pipeline.X_train, pipeline.y_train)
    trained_models["ensemble_bagging"] = bagging

print(f"\nSuccessfully trained {len(trained_models)} models!")

### 3. Evaluating Performance Across Standardized Metrics

In [ ]:
evaluator = ModelEvaluator()
results_df = evaluator.evaluate_all(trained_models, pipeline.X_test, pipeline.y_test)

metrics_display = results_df[["accuracy", "precision", "recall", "f1"]].sort_values(by="f1", ascending=False)
display(metrics_display)

# Plot F1 Comparison
fig = plot_model_comparison(results_df, metric="f1", title="Model F1-Score Comparison Across Algorithms")
plt.show()

### 4. PyTorch Deep Neural Network Inspection

In [ ]:
pytorch_model = trained_models.get("pytorch_dnn")
if pytorch_model and hasattr(pytorch_model, "training_losses_"):
    plt.figure(figsize=(8, 4))
    plt.plot(pytorch_model.training_losses_, marker='o', color='purple', label='Training Loss')
    plt.title('PyTorch Deep Neural Network Training Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('BCE Loss')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()